# 07 · Compare, disagree and adjudicate

**Spatial Humanities 2026 workshop**

Different methods often agree on easy cases and diverge exactly where scholarly judgement matters. This notebook makes disagreement visible and measures the amount of human intervention required.

## Learning goals
- distinguish model agreement from correctness;
- create a consensus annotation without hiding dissent;
- inspect missing-from/voter information;
- record human accept/edit/reject actions append-only;
- separate **review burden** from **correction burden**;
- produce tidy comparison records for later keynote figures.

> **Key message:** There is no single method winner across all dimensions.

In [ ]:
# Independent Colab setup.
import os, sys, json, subprocess, pathlib

REPO = "https://github.com/IgnatiusEzeani/spatio-textual.git"
BRANCH = "spatial-humanities-2026"

if not pathlib.Path("spatio-textual").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "spatio-textual"], check=True)
os.chdir("spatio-textual")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

print("Ready:", pathlib.Path.cwd())

## 1. A controlled disagreement example

We first use small deterministic records so the mechanics of adjudication are visible and reproducible. These are **teaching records**, not empirical model results.

In [ ]:
from spatio_textual.moe import ModelAnnotation, adjudicate_entities

text = "I travelled from Cambridge to London."
cam_start = text.index("Cambridge")
lon_start = text.index("London")

cambridge = {"text": "Cambridge", "label": "GPE", "start_char": cam_start, "end_char": cam_start + len("Cambridge")}
london = {"text": "London", "label": "GPE", "start_char": lon_start, "end_char": lon_start + len("London")}

records = [
    ModelAnnotation("method_A", {"text": text, "entities": [cambridge, london], "telemetry": []}),
    ModelAnnotation("method_B", {"text": text, "entities": [cambridge, london], "telemetry": []}),
    ModelAnnotation("method_C", {"text": text, "entities": [london], "telemetry": []}),
]

adj = adjudicate_entities(records, threshold=2/3)
print("Consensus:")
print(json.dumps(adj.consensus["entities"], indent=2))
print("\nDisagreements:")
print(json.dumps(adj.disagreements, indent=2))

`Cambridge` can enter the consensus at a two-of-three threshold while still remaining a disagreement because one method missed it. The consensus record therefore retains:

- `vote_count`
- `vote_ratio`
- `voters`
- `missing_from`
- `requires_review`

Consensus should not erase dissent.

## 2. Agreement is not correctness

Three systems can agree and still be wrong because they share training data, gazetteers, assumptions or ontology. Conversely, a minority method may identify something the others cannot represent.

For Spatial Humanities, disagreement is useful diagnostic evidence about:
- span boundaries;
- label inventories;
- ambiguous place resolution;
- implicit relations;
- historical names;
- inferred journey fields.

## 3. Human review as an auditable operation

The SH2026 common schema stores human decisions alongside machine outputs. Accepting, editing and rejecting are different events and should not be collapsed into a single "corrected" flag.

In [ ]:
from spatio_textual.review import apply_human_review, human_correction_burden

machine_journeys = [
    {
        "journeyId": "j1", "start_location": "Cambridge", "end_location": "London",
        "human_status": "unreviewed", "human_edits": [], "requires_review": True,
    },
    {
        "journeyId": "j2", "start_location": "Cambridge", "end_location": "London",
        "human_status": "unreviewed", "human_edits": [], "requires_review": True,
    },
    {
        "journeyId": "j3", "start_location": "Cambridge", "end_location": "Paris",
        "human_status": "unreviewed", "human_edits": [], "requires_review": True,
    },
]

accepted = apply_human_review(machine_journeys[0], action="accept", reason="human_flag")
edited = apply_human_review(
    machine_journeys[1], action="edit", field="end_location", new_value="London, England", reason="disambiguation"
)
rejected = apply_human_review(machine_journeys[2], action="reject", reason="unsupported_llm_field")

reviewed = [accepted, edited, rejected]
print(json.dumps(reviewed, indent=2, ensure_ascii=False))

The edit event preserves both the old and new values. This is important for reproducibility: the final clean value alone does not tell us how much human work was needed to obtain it.

In [ ]:
burden = human_correction_burden(reviewed)
print(json.dumps(burden, indent=2))

### Review burden vs correction burden

- **Review burden**: how many machine suggestions a human had to inspect.
- **Correction burden**: how many inspected suggestions had to be edited or rejected.

A system that is highly accurate but flags everything may still impose a high review burden. A system with low review volume but frequent edits may impose a high correction burden.

This distinction will be one of the keynote benchmark dimensions.

## 4. Side-by-side method comparison

Not every method should be forced into the same metric. Use `null` when a metric does not apply rather than writing zero.

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "example_id": "demo-01", "method": "manual_reference", "backend": "human", "model": None,
        "task": "spatial_annotation", "precision": None, "recall": None, "f1": None,
        "coverage": 1.0, "unsupported_rate": None, "ambiguous_rate": None,
        "human_edits_required": None, "latency_ms": None, "cost_usd_est": None,
        "notes": "Reference judgement; not assumed infallible."
    },
    {
        "example_id": "demo-01", "method": "rule_gazetteer", "backend": "rules", "model": "entity_ruler+regex",
        "task": "spatial_annotation", "precision": None, "recall": None, "f1": None,
        "coverage": None, "unsupported_rate": None, "ambiguous_rate": None,
        "human_edits_required": None, "latency_ms": None, "cost_usd_est": 0.0,
        "notes": "Populate with measured benchmark values only."
    },
    {
        "example_id": "demo-01", "method": "contextual_ner", "backend": "spacy", "model": "runtime-recorded",
        "task": "toponym_recognition", "precision": None, "recall": None, "f1": None,
        "coverage": None, "unsupported_rate": None, "ambiguous_rate": None,
        "human_edits_required": None, "latency_ms": None, "cost_usd_est": 0.0,
        "notes": "Named-entity ontology only; representational reach reported separately."
    },
    {
        "example_id": "demo-01", "method": "llm_journey", "backend": "llm", "model": "runtime-recorded",
        "task": "journey_extraction", "precision": None, "recall": None, "f1": None,
        "coverage": None, "unsupported_rate": None, "ambiguous_rate": None,
        "human_edits_required": None, "latency_ms": None, "cost_usd_est": None,
        "notes": "Requires evidence-grounding and inference audit."
    },
])

display(comparison)

The empty cells are deliberate. We will fill them only after the held-out benchmark is frozen and the relevant method is actually run. This prevents example-driven development from turning into accidental benchmark reporting.

## 5. A lightweight adjudication exercise

Choose one disagreement and record your decision. The objective is not to maximize agreement with the machines; it is to make the scholarly decision explicit.

In [ ]:
exercise = pd.DataFrame([
    {
        "item": "Cambridge",
        "method_A": "GPE",
        "method_B": "GPE",
        "method_C": "missing",
        "your_decision": "",
        "reason": "",
    }
])
exercise

Questions to discuss:

1. Would your decision change if the text were eighteenth-century rather than contemporary?
2. Would your decision change if the next sentence mentioned Massachusetts?
3. Is majority vote a method of truth, or merely a routing heuristic?
4. Which disagreements deserve mandatory human review?

## 6. Export the comparison skeleton

This tidy structure is designed to feed the later benchmark runner, Streamlit comparison view and keynote figures.

In [ ]:
from pathlib import Path

out_dir = Path("sh2026_outputs/comparisons")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "comparison_template.csv"
comparison.to_csv(out_path, index=False)
print(out_path)

## 7. Take-away

A hybrid workflow is not simply "use many models". It needs explicit rules for:

**what is compared → how disagreement is represented → when a human intervenes → how that intervention is recorded → how burden is measured**

This makes human review part of the experimental design rather than an invisible clean-up step.

**Next:** move from annotations to maps and other spatial representations while preserving uncertainty and provenance.